In [2]:
import pandas as pd
from tqdm import tqdm
import seaborn as sns

## Access TAMA)

In [3]:
ref=pd.read_pickle("all_runs/tama_liftoff_sqanti.pkl")

In [4]:
ref["structural_category"].value_counts()

full-splice_match          2226125
incomplete-splice_match     545989
novel_not_in_catalog        296930
novel_in_catalog            257671
intergenic                   61252
genic                        39882
antisense                    17112
fusion                       14837
genic_intron                    34
Name: structural_category, dtype: int64

In [5]:
# load full df
ref=pd.read_pickle("all_runs/tama_liftoff_sqanti.pkl")
# add new column
ref["cat_assign"]="None"
# and fill new colum based on 4 categories
ref.loc[(ref["structural_category"] != "antisense") & (ref["structural_category"] != "fusion") & (ref["structural_category"] != "intergenic"), "cat_assign"]="one2one"
ref.loc[(ref["structural_category"] == "antisense"), "cat_assign"]="antisense"
ref.loc[(ref["structural_category"] == "fusion"), "cat_assign"]="fusion"
ref.loc[((ref["structural_category"] == "intergenic") | (ref["structural_category"] == "genic_intron")), "cat_assign"]="novel"

# in some cases, fusion transcripts were reported as one2one if they had perfect splice junctions over their fusion genes.
# filter them and add replace the assignment
ref["association"] = ref["associated_gene"].str.split("_").str.len()
ref.loc[(ref["cat_assign"]=="one2one")&(ref["association"]>1), "cat_assign"] = "fusion"

# add gene read support
# cat_read_support is the number of reads supporting the individual category assignments of a gene (can be different in terms of clear, fusion and antisense)
# gene_association_read_support: sometimes there are multiple clear assignments within a gene, caused by a fusion gene also associated with the reads covering
# multiple ref genes. This number gives the support for each clear-case gene in the reference.
ref["cat_read_support"]=ref.groupby(by=["cat_assign","accession","gene"])["isoform_read_support"].transform("sum")
ref["gene_association_read_support"]=ref.groupby(by=["cat_assign","accession","gene","associated_gene"])["isoform_read_support"].transform("sum")

## Prepare Fusion genes for both type assignment and later detailed analysis (fusion type)

In [6]:
def liftoff_refs(df):
    # get_df
    accessions=df["accession"].unique().tolist()

    busco = pd.read_csv("/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/busco_Araport11_20221221.txt", header=None)
    house = pd.read_csv("/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/arat_housekeeping.txt", header=None)
    nlr = pd.read_csv("/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/nlrs_in_orthogroups_20221221.txt", header=None)

    busco["type"] = "busco"
    house["type"] = "housekeeping"
    nlr["type"] = "nlr"
    gene_type = pd.concat([busco,house,nlr])
    gene_type.columns = ["ref_isoform","type"]
    gene_type = gene_type.groupby("ref_isoform")["type"].apply(', '.join).reset_index()

    gene_type["ref_gene"] = gene_type["ref_isoform"].str.split(".").str[0]
    gene_type.loc[gene_type["type"] == "busco, nlr"] = "nlr"
    gene_type.loc[gene_type["type"] == "housekeeping, nlr"] = "nlr"
    gene_type.loc[gene_type["type"] == "busco, housekeeping"] = "housekeeping"
        
    gene_ref = pd.merge(left=df, right=gene_type, how='left', left_on='associated_gene', right_on='ref_gene')
    
    return gene_ref

In [7]:
def liftoff_refs_fusion(df):
    # get_df
    accessions=df["accession"].unique().tolist()

    busco = pd.read_csv("/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/busco_Araport11_20221221.txt", header=None)
    house = pd.read_csv("/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/arat_housekeeping.txt", header=None)
    nlr = pd.read_csv("/ebio/abt6_projects7/dliso/dlis/leon_pipeline/analysis/sqanti/ref_genes/nlrs_in_orthogroups_20221221.txt", header=None)

    busco["type"] = "busco"
    house["type"] = "housekeeping"
    nlr["type"] = "nlr"
    gene_type = pd.concat([busco,house,nlr])
    gene_type.columns = ["ref_isoform","type"]
    gene_type = gene_type.groupby("ref_isoform")["type"].apply(', '.join).reset_index()

    gene_type["ref_gene"] = gene_type["ref_isoform"].str.split(".").str[0]
    gene_type.loc[gene_type["type"] == "busco, nlr"] = "nlr"
    gene_type.loc[gene_type["type"] == "housekeeping, nlr"] = "nlr"
    gene_type.loc[gene_type["type"] == "busco, housekeeping"] = "housekeeping"
        
    gene_ref = pd.merge(left=df, right=gene_type, how='left', left_on='gene_split', right_on='ref_gene')
    
    return gene_ref

In [8]:
fus=ref.loc[ref["cat_assign"]=="fusion"].copy()
fus["gene_split"]=fus["associated_gene"].str.split("_")

In [9]:
split=fus.explode("gene_split")

In [10]:
# test for novel genes in fusion genes
for i in split["gene_split"].tolist():
    if i == "novelGene":
        print(i)

In [11]:
fusion_types=liftoff_refs_fusion(split).drop(columns=["ref_isoform","ref_gene"])
fusion_types["type"] = fusion_types["type"].fillna("other")

In [12]:
fusion_types.reset_index(drop=True, inplace=True)

In [13]:
fusion_out=pd.DataFrame()
for i in fusion_types.groupby(by=["accession", "gene", "isoform"], as_index=False):
    t=i[1]["type"].value_counts()
    t=t.reset_index()

    t=t["index"].tolist()

    i[1]["fusion_order"] = "_".join(i[1]["type"].tolist())
    
    if "busco" in t:
        i[1]["busco"] = 1
    if "nlr" in t:
        i[1]["nlr"] = 1
    if "housekeeping" in t:
        i[1]["housekeeping"] = 1
    if "other" in t:
        i[1]["other"] = 1

    fusion_out=pd.concat([fusion_out,i[1]], axis=0)

In [14]:
fusion_out

,gene,isoform,accession,associated_gene,associated_transcript,exons,length,isoform_read_support,gene_read_support,structural_category,...,association,cat_read_support,gene_association_read_support,gene_split,type,fusion_order,other,busco,housekeeping,nlr
1661,G10006,G10006.5,at6923,AT3G58700_AT3G58690,novel,10,2461,1,11,fusion,...,2,1,1,AT3G58700,other,other_other,1.0,NaN,NaN,NaN
1662,G10006,G10006.5,at6923,AT3G58700_AT3G58690,novel,10,2461,1,11,fusion,...,2,1,1,AT3G58690,other,other_other,1.0,NaN,NaN,NaN
1663,G10013,G10013.2,at6923,AT3G58810_AT3G58800,AT3G58800.1,2,1563,1,5,full-splice_match,...,2,3,3,AT3G58810,other,other_other,1.0,NaN,NaN,NaN
1664,G10013,G10013.2,at6923,AT3G58810_AT3G58800,AT3G58800.1,2,1563,1,5,full-splice_match,...,2,3,3,AT3G58800,other,other_other,1.0,NaN,NaN,NaN
1665,G10013,G10013.3,at6923,AT3G58810_AT3G58800,AT3G58800.1,2,3552,1,5,full-splice_match,...,2,3,3,AT3G58810,other,other_other,1.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44635,G9951,G9951.9,col0,AT4G00030_AT4G00040,AT4G00030.1,3,2332,1,21,full-splice_match,...,2,19,19,AT4G00040,other,busco_other,1.0,1.0,NaN,NaN
44636,G9987,G9987.2,col0,AT4G00570_AT4G00560,novel,23,3761,1,71,fusion,...,2,1,1,AT4G00570,other,other_busco,1.0,1.0,NaN,NaN
44637,G9987,G9987.2,col0,AT4G00570_AT4G00560,novel,23,3761,1,71,fusion,...,2,1,1,AT4G00560,busco,other_busco,1.0,1.0,NaN,NaN
44638,G9989,G9989.11,col0,AT4G00620_AT4G00630,novel,8,1542,1,88,fusion,...,2,1,1,AT4G00620,other,other_other,1.0,NaN,NaN,NaN


In [15]:
fusion_types= fusion_out.drop(columns=["gene_split","type","association"]).drop_duplicates()
fusion_types.fillna({'other':0, 'busco':0, 'nlr':0,'housekeeping':0}, inplace=True)

## Connect all ref gene sets (Clear Cases + Fusion genes (Order: NLR>House>Busco>Other)) - This creates the DF for subsequent analysis

### The reason to do it this preciesely for the reference groups to have a fair comparison between them and add duplicate fusion genes to each of the potential groups it belongs. In the antisense case, it was done quick without distinguishing and just keeping the order above.

In [16]:
NLRs=pd.DataFrame()
for acc in ref["accession"].unique().tolist():
    
    clear_isos=ref.loc[(ref["accession"]==acc)&(ref["cat_assign"] == "one2one")]
    isos_type=liftoff_refs(clear_isos).drop(columns=["ref_isoform","ref_gene"])
    nlr_isos=isos_type.loc[isos_type["type"]=="nlr"]

    
    b=fusion_types.loc[(fusion_types["nlr"]==1)&(fusion_types["accession"]==acc)]["isoform"].tolist()
    fusion_isos=ref.loc[(ref["accession"]==acc)&(ref["isoform"].isin(b))]

    c=pd.concat([nlr_isos,fusion_isos], ignore_index=True)
    
    c["isoforms_per_gene"]= c.groupby(by="gene")["isoform"].transform(lambda x: len(x.tolist()))
    c["ORFs_per_gene"]= c.groupby(by=["gene"])["Sqanti_ORF_seq"].transform(lambda x: len(set(x.tolist())))
    c["isoforms_per_ORFs"]= c.groupby(by=["gene","Sqanti_ORF_seq"])["isoform"].transform(lambda x: len(x.tolist()))
    
    #print("Mean nlr isoforms per gene:")
    hipg=c[["gene","isoforms_per_gene"]].drop_duplicates()
    #print(hipg["isoforms_per_gene"].apply("mean"))

    #print("Mean nlr ORFs per gene:")
    hopg=c[["gene","ORFs_per_gene"]].drop_duplicates()
    #print(hopg["ORFs_per_gene"].apply("mean"))
    #print("\n")
    
    
    NLRs=pd.concat([NLRs,c])
NLRs["type"]="nlr"

In [17]:
House=pd.DataFrame()
for acc in ref["accession"].unique().tolist():
    
    clear_isos=ref.loc[(ref["accession"]==acc)&(ref["cat_assign"] == "one2one")]
    isos_type=liftoff_refs(clear_isos).drop(columns=["ref_isoform","ref_gene"])
    house_isos=isos_type.loc[isos_type["type"]=="housekeeping"]

    
    b=fusion_types.loc[(fusion_types["housekeeping"]==1)&(fusion_types["accession"]==acc)]["isoform"].tolist()
    fusion_isos=ref.loc[(ref["accession"]==acc)&(ref["isoform"].isin(b))]

    c=pd.concat([house_isos,fusion_isos], ignore_index=True)
    #c=ref.loc[(ref["accession"]==f"{acc}") & (ref["gene"].isin(set(a[f"{acc}"].tolist() + b[f"{acc}"].tolist())))&~(ref["cat_assign"].isin(["antisense","novel"]))].copy()
    
    c["isoforms_per_gene"]= c.groupby(by="gene")["isoform"].transform(lambda x: len(x.tolist()))
    c["ORFs_per_gene"]= c.groupby(by=["gene"])["Sqanti_ORF_seq"].transform(lambda x: len(set(x.tolist())))
    c["isoforms_per_ORFs"]= c.groupby(by=["gene","Sqanti_ORF_seq"])["isoform"].transform(lambda x: len(x.tolist()))
    
    #print("Mean nlr isoforms per gene:")
    hipg=c[["gene","isoforms_per_gene"]].drop_duplicates()
    #print(hipg["isoforms_per_gene"].apply("mean"))

    #print("Mean nlr ORFs per gene:")
    hopg=c[["gene","ORFs_per_gene"]].drop_duplicates()
    #print(hopg["ORFs_per_gene"].apply("mean"))
    #print("\n")
    
    
    House=pd.concat([House,c])
House["type"]="housekeeping"

In [18]:
Busco=pd.DataFrame()
for acc in ref["accession"].unique().tolist():
    
    clear_isos=ref.loc[(ref["accession"]==acc)&(ref["cat_assign"] == "one2one")]
    isos_type=liftoff_refs(clear_isos).drop(columns=["ref_isoform","ref_gene"])
    busco_isos=isos_type.loc[isos_type["type"]=="busco"]

    
    b=fusion_types.loc[(fusion_types["busco"]==1)&(fusion_types["accession"]==acc)]["isoform"].tolist()
    fusion_isos=ref.loc[(ref["accession"]==acc)&(ref["isoform"].isin(b))]

    c=pd.concat([busco_isos,fusion_isos], ignore_index=True)
    #c=ref.loc[(ref["accession"]==f"{acc}") & (ref["gene"].isin(set(a[f"{acc}"].tolist() + b[f"{acc}"].tolist())))&~(ref["cat_assign"].isin(["antisense","novel"]))].copy()
    
    c["isoforms_per_gene"]= c.groupby(by="gene")["isoform"].transform(lambda x: len(x.tolist()))
    c["ORFs_per_gene"]= c.groupby(by=["gene"])["Sqanti_ORF_seq"].transform(lambda x: len(set(x.tolist())))
    c["isoforms_per_ORFs"]= c.groupby(by=["gene","Sqanti_ORF_seq"])["isoform"].transform(lambda x: len(x.tolist()))
    
    #print("Mean nlr isoforms per gene:")
    hipg=c[["gene","isoforms_per_gene"]].drop_duplicates()
    #print(hipg["isoforms_per_gene"].apply("mean"))

    #print("Mean nlr ORFs per gene:")
    hopg=c[["gene","ORFs_per_gene"]].drop_duplicates()
    #print(hopg["ORFs_per_gene"].apply("mean"))
    #print("\n")
    
    
    Busco=pd.concat([Busco,c])
Busco["type"]="busco"

In [19]:
Other=pd.DataFrame()
for acc in ref["accession"].unique().tolist():
    
    clear_isos=ref.loc[(ref["accession"]==acc)&(ref["cat_assign"] == "one2one")]
    isos_type=liftoff_refs(clear_isos).drop(columns=["ref_isoform","ref_gene"])
    isos_type["type"] = isos_type["type"].fillna("other")
    other_isos=isos_type.loc[isos_type["type"]=="other"]

    
    b=fusion_types.loc[(fusion_types["other"]==1)&(fusion_types["accession"]==acc)]["isoform"].tolist()
    fusion_isos=ref.loc[(ref["accession"]==acc)&(ref["isoform"].isin(b))]

    c=pd.concat([other_isos,fusion_isos], ignore_index=True)
    
    c["isoforms_per_gene"]= c.groupby(by="gene")["isoform"].transform(lambda x: len(x.tolist()))
    c["ORFs_per_gene"]= c.groupby(by=["gene"])["Sqanti_ORF_seq"].transform(lambda x: len(set(x.tolist())))
    c["isoforms_per_ORFs"]= c.groupby(by=["gene","Sqanti_ORF_seq"])["isoform"].transform(lambda x: len(x.tolist()))
    
    #print("Mean nlr isoforms per gene:")
    hipg=c[["gene","isoforms_per_gene"]].drop_duplicates()
    #print(hipg["isoforms_per_gene"].apply("mean"))

    #print("Mean nlr ORFs per gene:")
    hopg=c[["gene","ORFs_per_gene"]].drop_duplicates()
    #print(hopg["ORFs_per_gene"].apply("mean"))
    #print("\n")
    
    
    Other=pd.concat([Other,c])
Other["type"]="other"

In [20]:
all_types=pd.concat([NLRs,Busco,House,Other], ignore_index=True)
all_types

,gene,isoform,accession,associated_gene,associated_transcript,exons,length,isoform_read_support,gene_read_support,structural_category,subcategory,Sqanti_ORF_seq,cat_assign,association,cat_read_support,gene_association_read_support,type,isoforms_per_gene,ORFs_per_gene,isoforms_per_ORFs
0,G1128,G1128.10,at6923,AT1G15890,AT1G15890.1,1,3275,1,59,full-splice_match,mono-exon,MGNCVSLDVSCHHQTLNHACGCLFGDGNYIHMMEANLEALEKTIQE...,one2one,1,59,13,nlr,12,2,9.0
1,G1128,G1128.11,at6923,AT1G15890,AT1G15890.1,1,2744,1,59,full-splice_match,mono-exon,MDGVLTFTVKMHDVIREMALWIASKFGKQKETLCVRSGAQLRQIPK...,one2one,1,59,13,nlr,12,2,3.0
2,G1128,G1128.12,at6923,AT1G15890,AT1G15890.1,1,2795,1,59,full-splice_match,mono-exon,MDGVLTFTVKMHDVIREMALWIASKFGKQKETLCVRSGAQLRQIPK...,one2one,1,59,13,nlr,12,2,3.0
3,G1128,G1128.13,at6923,AT1G15890,AT1G15890.1,1,1821,1,59,full-splice_match,mono-exon,MDGVLTFTVKMHDVIREMALWIASKFGKQKETLCVRSGAQLRQIPK...,one2one,1,59,13,nlr,12,2,3.0
4,G1128,G1128.2,at6923,AT1G15890,novel,2,3331,1,59,genic,multi-exon,MGNCVSLDVSCHHQTLNHACGCLFGDGNYIHMMEANLEALEKTIQE...,one2one,1,59,13,nlr,12,2,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3410918,G16302,G16302.24,col0,ATCG01300_ATCG01310,novel,2,818,1,136,fusion,multi-exon,MAIHLYKTSTPSTRNGAVDSQVKSNPRNNLICGQHHCGKGRNARGI...,fusion,2,15,15,other,49,8,2.0
3410919,G16302,G16302.26,col0,ATCG01300_ATCG01310,novel,2,831,2,136,fusion,multi-exon,MAIHLYKTSTPSTRNGAVDSQVKSNPRNNLICGQHHCGKGRNARGI...,fusion,2,15,15,other,49,8,4.0
3410920,G16302,G16302.5,col0,ATCG01300_ATCG01310,novel,2,1204,1,136,fusion,multi-exon,MAIHLYKTSTPSTRNGAVDSQVKSNPRNNLICGQHHCGKGRNARGI...,fusion,2,15,15,other,49,8,4.0
3410921,G16302,G16302.7,col0,ATCG01300_ATCG01310,novel,2,1141,6,136,fusion,multi-exon,MAIHLYKTSTPSTRNGAVDSQVKSNPRNNLICGQHHCGKGRNARGI...,fusion,2,15,15,other,49,8,4.0


In [21]:
drop=[]
for i in tqdm(all_types.groupby("accession")):
    df = i[1]
    boolean=df["isoform"].value_counts()>1
    boolean=boolean.rename_axis('isoform').reset_index(name='bool')
    #print(boolean.loc[boolean["bool"]==True])
    #print(boolean)
    dups = boolean.loc[boolean["bool"]==True]["isoform"].tolist()
    for iso in dups:
        iso_df=df[df["isoform"]==iso]
        if "nlr" in iso_df["type"].tolist():
            drop.extend(iso_df[~(iso_df["type"]=="nlr")].index.tolist())
            continue
        if "housekeeping" in iso_df["type"].tolist():
            drop.extend(iso_df[~(iso_df["type"]=="housekeeping")].index.tolist())
            continue
        if "busco" in iso_df["type"].tolist():
            drop.extend(iso_df[~(iso_df["type"]=="busco")].index.tolist())
            continue

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 18/18 [03:28<00:00, 11.59s/it]


In [22]:
all_types=all_types.drop(drop).drop_duplicates()

In [25]:
all_types

,gene,isoform,accession,associated_gene,associated_transcript,exons,length,isoform_read_support,gene_read_support,structural_category,subcategory,Sqanti_ORF_seq,cat_assign,association,cat_read_support,gene_association_read_support,type,isoforms_per_gene,ORFs_per_gene,isoforms_per_ORFs
0,G1128,G1128.10,at6923,AT1G15890,AT1G15890.1,1,3275,1,59,full-splice_match,mono-exon,MGNCVSLDVSCHHQTLNHACGCLFGDGNYIHMMEANLEALEKTIQE...,one2one,1,59,13,nlr,12,2,9.0
1,G1128,G1128.11,at6923,AT1G15890,AT1G15890.1,1,2744,1,59,full-splice_match,mono-exon,MDGVLTFTVKMHDVIREMALWIASKFGKQKETLCVRSGAQLRQIPK...,one2one,1,59,13,nlr,12,2,3.0
2,G1128,G1128.12,at6923,AT1G15890,AT1G15890.1,1,2795,1,59,full-splice_match,mono-exon,MDGVLTFTVKMHDVIREMALWIASKFGKQKETLCVRSGAQLRQIPK...,one2one,1,59,13,nlr,12,2,3.0
3,G1128,G1128.13,at6923,AT1G15890,AT1G15890.1,1,1821,1,59,full-splice_match,mono-exon,MDGVLTFTVKMHDVIREMALWIASKFGKQKETLCVRSGAQLRQIPK...,one2one,1,59,13,nlr,12,2,3.0
4,G1128,G1128.2,at6923,AT1G15890,novel,2,3331,1,59,genic,multi-exon,MGNCVSLDVSCHHQTLNHACGCLFGDGNYIHMMEANLEALEKTIQE...,one2one,1,59,13,nlr,12,2,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3410918,G16302,G16302.24,col0,ATCG01300_ATCG01310,novel,2,818,1,136,fusion,multi-exon,MAIHLYKTSTPSTRNGAVDSQVKSNPRNNLICGQHHCGKGRNARGI...,fusion,2,15,15,other,49,8,2.0
3410919,G16302,G16302.26,col0,ATCG01300_ATCG01310,novel,2,831,2,136,fusion,multi-exon,MAIHLYKTSTPSTRNGAVDSQVKSNPRNNLICGQHHCGKGRNARGI...,fusion,2,15,15,other,49,8,4.0
3410920,G16302,G16302.5,col0,ATCG01300_ATCG01310,novel,2,1204,1,136,fusion,multi-exon,MAIHLYKTSTPSTRNGAVDSQVKSNPRNNLICGQHHCGKGRNARGI...,fusion,2,15,15,other,49,8,4.0
3410921,G16302,G16302.7,col0,ATCG01300_ATCG01310,novel,2,1141,6,136,fusion,multi-exon,MAIHLYKTSTPSTRNGAVDSQVKSNPRNNLICGQHHCGKGRNARGI...,fusion,2,15,15,other,49,8,4.0


In [23]:
# This is the fall_typesdataframe to continue 
all_types
all_types.to_pickle("clear-fusion/clear-fusion_tama_liftoff.pkl")
#all_types.to_pickle("../5_NLRs/clear-fusion_tama_liftoff.pkl")
#all_types.to_pickle("../3_fusion_genes/clear-fusion_tama_liftoff.pkl")